# Premier League — Entrenamiento del Modelo
**Objetivo:** Cargar datos, construir features, entrenar XGBoost y guardar `modelo_premier.pkl`

Correr todas las celdas de arriba a abajo. Al finalizar tendrás `modelo_premier.pkl` listo para usar en el app notebook.

In [1]:
# ─────────────────────────────────────────────
# CELDA 1 — Imports
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

print('Imports OK')

Imports OK


In [2]:
# ─────────────────────────────────────────────
# CELDA 2 — Cargar datos (6 temporadas)
# ─────────────────────────────────────────────
temporadas = ['2021', '2122', '2223', '2324', '2425', '2526']
base_url   = 'https://www.football-data.co.uk/mmz4281/{}/E0.csv'

dfs = []
for t in temporadas:
    df_temp = pd.read_csv(base_url.format(t))
    df_temp['season'] = t
    dfs.append(df_temp)

df_raw = pd.concat(dfs, ignore_index=True)

cols = ['season', 'Date', 'HomeTeam', 'AwayTeam',
        'FTHG', 'FTAG', 'FTR', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC']

df = df_raw[cols].dropna(subset=['FTR'])
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date').reset_index(drop=True)

print(f'Total partidos: {len(df)}')
print(f'Rango fechas:   {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'\nDistribución resultados:')
print(df['FTR'].value_counts())

Total partidos: 2209
Rango fechas:   2020-09-12 → 2026-03-22

Distribución resultados:
FTR
H    950
A    742
D    517
Name: count, dtype: int64


In [3]:
# ─────────────────────────────────────────────
# CELDA 3 — Funciones de features
# ─────────────────────────────────────────────

def get_team_stats_v2(team, date, df, n_recent=5):
    """Stats combinadas: forma reciente (últimos 5) + temporada actual."""
    home_all = df[(df['HomeTeam'] == team) & (df['Date'] < date)]
    away_all = df[(df['AwayTeam'] == team) & (df['Date'] < date)]
    all_games = pd.concat([home_all, away_all]).sort_values('Date')

    if len(all_games) < 3:
        return None

    current_season = df[df['Date'] < date]['season'].iloc[-1]
    home_szn  = home_all[home_all['season'] == current_season]
    away_szn  = away_all[away_all['season'] == current_season]
    season_games = pd.concat([home_szn, away_szn]).sort_values('Date')
    recent = all_games.tail(n_recent)

    def calc_stats(games, team):
        if len(games) == 0:
            return None
        gf=gc=sot_f=sot_c=wins=draws=home_wins=home_games=0
        for _, r in games.iterrows():
            is_home = r['HomeTeam'] == team
            gf    += r['FTHG'] if is_home else r['FTAG']
            gc    += r['FTAG'] if is_home else r['FTHG']
            sot_f += r['HST']  if is_home else r['AST']
            sot_c += r['AST']  if is_home else r['HST']
            won  = (is_home and r['FTR']=='H') or (not is_home and r['FTR']=='A')
            drew = r['FTR'] == 'D'
            if won:  wins  += 1
            if drew: draws += 1
            if is_home:
                home_games += 1
                if r['FTR'] == 'H': home_wins += 1
        m = len(games)
        return {
            'gf_pg':     gf/m,
            'gc_pg':     gc/m,
            'dif_goles': (gf-gc)/m,
            'sot_f_pg':  sot_f/m,
            'sot_c_pg':  sot_c/m,
            'win_rate':  wins/m,
            'draw_rate': draws/m,
            'home_wr':   home_wins/home_games if home_games > 0 else wins/m,
        }

    recent_s = calc_stats(recent, team)
    season_s = calc_stats(season_games, team) if len(season_games) >= 3 else recent_s

    if recent_s is None:
        return None

    combined = {}
    for k in recent_s:
        combined[f'recent_{k}'] = recent_s[k]
        combined[f'season_{k}'] = season_s[k] if season_s else recent_s[k]
    return combined


def get_h2h_stats(home_team, away_team, date, df, n=10):
    """Historial directo entre dos equipos — últimos n enfrentamientos."""
    h2h = df[
        ((df['HomeTeam'] == home_team) & (df['AwayTeam'] == away_team)) |
        ((df['HomeTeam'] == away_team) & (df['AwayTeam'] == home_team))
    ]
    h2h = h2h[h2h['Date'] < date].tail(n)

    if len(h2h) < 3:
        return {'h2h_home_wr': 0.33, 'h2h_draw_rate': 0.33, 'h2h_away_wr': 0.33}

    home_wins = ((h2h['HomeTeam'] == home_team) & (h2h['FTR'] == 'H')).sum() + \
                ((h2h['AwayTeam'] == home_team) & (h2h['FTR'] == 'A')).sum()
    draws     = (h2h['FTR'] == 'D').sum()
    away_wins = len(h2h) - home_wins - draws
    m = len(h2h)
    return {
        'h2h_home_wr':   home_wins / m,
        'h2h_draw_rate': draws / m,
        'h2h_away_wr':   away_wins / m,
    }


def get_tabla_posicion(team, date, df):
    """Posición y puntos en la tabla al momento del partido."""
    season = df[df['Date'] < date]['season'].iloc[-1]
    partidos_szn = df[(df['season'] == season) & (df['Date'] < date)]
    equipos = pd.concat([partidos_szn['HomeTeam'], partidos_szn['AwayTeam']]).unique()

    tabla = []
    for eq in equipos:
        home_p = partidos_szn[partidos_szn['HomeTeam'] == eq]
        away_p = partidos_szn[partidos_szn['AwayTeam'] == eq]
        pts = ((home_p['FTR']=='H').sum()*3 + (home_p['FTR']=='D').sum() +
               (away_p['FTR']=='A').sum()*3 + (away_p['FTR']=='D').sum())
        gf  = home_p['FTHG'].sum() + away_p['FTAG'].sum()
        gc  = home_p['FTAG'].sum() + away_p['FTHG'].sum()
        pj  = len(home_p) + len(away_p)
        tabla.append({'equipo': eq, 'pts': pts, 'gf': gf, 'gc': gc, 'pj': pj})

    tabla_df = pd.DataFrame(tabla).sort_values('pts', ascending=False).reset_index(drop=True)
    tabla_df['posicion']      = tabla_df.index + 1
    tabla_df['total_equipos'] = len(tabla_df)

    fila = tabla_df[tabla_df['equipo'] == team]
    if len(fila) == 0:
        return {'posicion': 10, 'pts_pg': 1.0, 'dif_goles_szn': 0, 'pct_posicion': 0.5}

    fila = fila.iloc[0]
    pj   = max(fila['pj'], 1)
    return {
        'posicion':      fila['posicion'],
        'pts_pg':        fila['pts'] / pj,
        'dif_goles_szn': (fila['gf'] - fila['gc']) / pj,
        'pct_posicion':  1 - (fila['posicion'] / fila['total_equipos']),
    }


print('Funciones definidas OK')

Funciones definidas OK


In [4]:
# ─────────────────────────────────────────────
# CELDA 4 — Construir dataset v3
# Tarda ~3-5 minutos
# ─────────────────────────────────────────────
rows_v3 = []

for idx, partido in df.iterrows():
    home_team = partido['HomeTeam']
    away_team = partido['AwayTeam']
    date      = partido['Date']
    season    = partido['season']

    home_stats = get_team_stats_v2(home_team, date, df)
    away_stats = get_team_stats_v2(away_team, date, df)

    if home_stats is None or away_stats is None:
        continue

    h2h        = get_h2h_stats(home_team, away_team, date, df)
    home_tabla = get_tabla_posicion(home_team, date, df)
    away_tabla = get_tabla_posicion(away_team, date, df)

    row = {
        'date':      date,
        'season':    season,
        'home_team': home_team,
        'away_team': away_team,
        'resultado': partido['FTR'],
    }

    for k, v in home_stats.items(): row[f'h_{k}'] = v
    for k, v in away_stats.items(): row[f'a_{k}'] = v
    for k, v in h2h.items():        row[k] = v
    for k, v in home_tabla.items(): row[f'h_tabla_{k}'] = v
    for k, v in away_tabla.items(): row[f'a_tabla_{k}'] = v

    row['dif_recent_wr']  = home_stats['recent_win_rate']  - away_stats['recent_win_rate']
    row['dif_season_wr']  = home_stats['season_win_rate']  - away_stats['season_win_rate']
    row['dif_recent_gf']  = home_stats['recent_gf_pg']     - away_stats['recent_gf_pg']
    row['dif_recent_gc']  = home_stats['recent_gc_pg']     - away_stats['recent_gc_pg']
    row['dif_recent_dif'] = home_stats['recent_dif_goles'] - away_stats['recent_dif_goles']
    row['dif_season_dif'] = home_stats['season_dif_goles'] - away_stats['season_dif_goles']
    row['home_advantage'] = home_stats['recent_home_wr']   - away_stats['recent_win_rate']
    row['dif_pts_pg']     = home_tabla['pts_pg']           - away_tabla['pts_pg']
    row['dif_posicion']   = away_tabla['posicion']         - home_tabla['posicion']

    rows_v3.append(row)

df_v3 = pd.DataFrame(rows_v3)

print(f'Partidos: {len(df_v3)}')
print(f'Features: {len(df_v3.columns) - 5}')
print(f'\nTemporadas:')
print(df_v3['season'].value_counts().sort_index())

Partidos: 2153
Features: 52

Temporadas:
season
2021    348
2122    371
2223    374
2324    377
2425    377
2526    306
Name: count, dtype: int64


In [5]:
# ─────────────────────────────────────────────
# CELDA 5 — Train/Test split temporal
# Train: temporadas 2021→2425
# Test:  temporada 2526
# ─────────────────────────────────────────────
train_v3 = df_v3[df_v3['season'] != '2526'].copy()
test_v3  = df_v3[df_v3['season'] == '2526'].copy()

feature_cols_v3 = [c for c in df_v3.columns
                   if c not in ['date', 'season', 'home_team', 'away_team', 'resultado']]

X_train_v3 = train_v3[feature_cols_v3]
X_test_v3  = test_v3[feature_cols_v3]

le = LabelEncoder()
y_train_v3 = le.fit_transform(train_v3['resultado'])
y_test_v3  = le.transform(test_v3['resultado'])

print(f'Train: {len(X_train_v3)} partidos ({train_v3["season"].min()} → {train_v3["season"].max()})')
print(f'Test:  {len(X_test_v3)} partidos (temporada 2526)')
print(f'Clases: {le.classes_}')
print(f'Features: {len(feature_cols_v3)}')

Train: 1847 partidos (2021 → 2425)
Test:  306 partidos (temporada 2526)
Clases: ['A' 'D' 'H']
Features: 52


In [6]:
# ─────────────────────────────────────────────
# CELDA 6 — Entrenar XGBoost + Calibrar
# ─────────────────────────────────────────────
xgb_v3 = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

xgb_v3.fit(X_train_v3, y_train_v3)

model_v3 = CalibratedClassifierCV(xgb_v3, cv=5, method='isotonic')
model_v3.fit(X_train_v3, y_train_v3)

print('Modelo entrenado y calibrado OK')

Modelo entrenado y calibrado OK


In [7]:
# ─────────────────────────────────────────────
# CELDA 7 — Evaluar modelo
# ─────────────────────────────────────────────
y_pred_v3       = model_v3.predict(X_test_v3)
y_pred_proba_v3 = model_v3.predict_proba(X_test_v3)

accuracy_v3 = accuracy_score(y_test_v3, y_pred_v3)
logloss_v3  = log_loss(y_test_v3, y_pred_proba_v3)

print(f'Accuracy:           {accuracy_v3:.1%}')
print(f'Log Loss:           {logloss_v3:.4f}')
print(f'Baseline (siempre H): 41.2%')
print(f'Mejora vs baseline: {(accuracy_v3 - 0.412):+.1%}')
print(f'\nReporte por clase:')
print(classification_report(y_test_v3, y_pred_v3, target_names=le.classes_))

# Accuracy por nivel de confianza
proba_df_v3 = pd.DataFrame(y_pred_proba_v3, columns=['prob_A', 'prob_D', 'prob_H'])
proba_df_v3['confianza'] = proba_df_v3[['prob_A', 'prob_D', 'prob_H']].max(axis=1)
proba_df_v3['pred']      = le.inverse_transform(y_pred_v3)
proba_df_v3['real']      = le.inverse_transform(y_test_v3)
proba_df_v3['correcto']  = proba_df_v3['pred'] == proba_df_v3['real']

bins   = [0.33, 0.40, 0.45, 0.50, 0.55, 1.0]
labels = ['33-40%', '40-45%', '45-50%', '50-55%', '>55%']
proba_df_v3['confianza_bin'] = pd.cut(proba_df_v3['confianza'], bins=bins, labels=labels)

print('\nAccuracy por confianza:')
print(proba_df_v3.groupby('confianza_bin', observed=True).agg(
    partidos=('correcto', 'count'),
    accuracy=('correcto', 'mean')
).round(3))

print(f'\nPartidos con confianza >55%: {(proba_df_v3["confianza"] > 0.55).sum()}')
print(f'Accuracy en esos partidos:   {proba_df_v3[proba_df_v3["confianza"] > 0.55]["correcto"].mean():.1%}')

Accuracy:           48.4%
Log Loss:           1.0337
Baseline (siempre H): 41.2%
Mejora vs baseline: +7.2%

Reporte por clase:
              precision    recall  f1-score   support

           A       0.43      0.57      0.49        96
           D       1.00      0.02      0.05        84
           H       0.51      0.72      0.60       126

    accuracy                           0.48       306
   macro avg       0.65      0.44      0.38       306
weighted avg       0.62      0.48      0.41       306


Accuracy por confianza:
               partidos  accuracy
confianza_bin                    
33-40%               32     0.438
40-45%               79     0.418
45-50%               61     0.426
50-55%               41     0.512
>55%                 93     0.581

Partidos con confianza >55%: 93
Accuracy en esos partidos:   58.1%


In [ ]:
# ─────────────────────────────────────────────
# CELDA 8 — Guardar modelo en disco
# Genera: modelo_premier.pkl
# ─────────────────────────────────────────────
# with open('modelo_premier.pkl', 'wb') as f:
#     pickle.dump({
#         'model_v3':        model_v3,
#         'feature_cols_v3': feature_cols_v3,
#         'le':              le,
#         'df':              df,
#     }, f)

# print('modelo_premier.pkl guardado correctamente')
# print(f'Features guardados: {len(feature_cols_v3)}')
# print('\nListo — abre premier_league_app.ipynb para predecir partidos')

# ─────────────────────────────────────────────
# CELDA 8 — Guardar modelo con joblib - modelo en disco
# Genera: modelo_premier.pkl
# ─────────────────────────────────────────────
import joblib

joblib.dump({
    'model_v3':        model_v3,
    'feature_cols_v3': feature_cols_v3,
    'le':              le,
    'df':              df,
}, 'modelo_premier.pkl')

print('modelo_premier.pkl guardado con joblib')

modelo_premier.pkl guardado correctamente
Features guardados: 52

Listo — abre premier_league_app.ipynb para predecir partidos
